In [67]:
import pandas as pd
import numpy as np
import xarray as xr
from shapely.geometry import Point
import geopandas as gpd
from geopy.distance import geodesic
from sklearn.neighbors import BallTree
from scipy.ndimage import gaussian_gradient_magnitude
from scipy.spatial import cKDTree
from shapely.ops import nearest_points
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix

In [11]:
data=pd.read_csv(r'data_renew.csv')
data2=pd.read_excel(r'input.xlsx')

In [12]:
data=data.drop(columns=['Hook'])
data.columns=['Date','Latitude','Longitude','Total']
data2=data2.rename(columns={'CATCH_DATE':'Date','WEIGHT (Kg)':'Total','lat':'Latitude','lng':'Longitude'})
data2=data2[['Date','Latitude','Longitude','Total']]

In [13]:
data.info()
data2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3689 entries, 0 to 3688
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       3689 non-null   object 
 1   Latitude   3689 non-null   float64
 2   Longitude  3689 non-null   float64
 3   Total      3669 non-null   float64
dtypes: float64(3), object(1)
memory usage: 115.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3424 entries, 0 to 3423
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       3424 non-null   datetime64[ns]
 1   Latitude   3424 non-null   float64       
 2   Longitude  3424 non-null   float64       
 3   Total      3424 non-null   int64         
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 107.1 KB


In [14]:
data['Date']=pd.to_datetime(data['Date'])
data['Date'].dt.year.unique()

array([2009, 2008, 2007, 2010, 2011, 2006, 2012, 2013, 2014, 2015, 2016,
       2017], dtype=int32)

In [15]:
data = data[
    data['Latitude'].between(-90, 90) &
    data['Longitude'].between(-180, 180)
]
data2 = data2[
    data2['Latitude'].between(-90, 90) &
    data2['Longitude'].between(-180, 180)
]

In [16]:
import pandas as pd
from global_land_mask import globe

is_land = globe.is_land(
    data["Latitude"].values,
    data["Longitude"].values
)
is_land_2 = globe.is_land(
    data2["Latitude"].values,
    data2["Longitude"].values
)
print("data_1")
print("Total points :", len(data))
print("Land points  :", is_land.sum())
print("Ocean points :", (~is_land).sum())
print("data_2")
print("Total points :", len(data2))
print("Land points  :", is_land_2.sum())
print("Ocean points :", (~is_land_2).sum())

data = data[~is_land]
data2 = data2[~is_land_2]

data_1
Total points : 3689
Land points  : 112
Ocean points : 3577
data_2
Total points : 3424
Land points  : 3
Ocean points : 3421


In [17]:
nc_sst=xr.open_dataset(r'C3S-GLO-SST-L4-REP-OBS-SST_analysed_sst_65.02E-95.97E_5.03N-23.98N_1998-01-10-2023-02-25.nc')
nc_curr=xr.open_dataset(r'cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_ugos-vgos-sla_65.06E-95.94E_5.06N-23.94N_1998-01-10-2023-02-25.nc')
chl_nc = xr.open_dataset(r'cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D_CHL_65.02E-95.98E_5.02N-23.98N_1998-01-01-2023-02-25.nc')
nc_cca_mag = xr.open_dataset(r'cca_magnitude_zero_2.nc')
wind=xr.open_dataset(r'era5_wind_all.nc')
nc_depth=xr.open_dataset(r'gebco_2026_n24.0_s2.0_w65.0_e97.0.nc')
nc_sss = xr.open_dataset(
    "cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos_66.56E-95.56E_2.44N-23.06N_0.00m_1998-01-01-2023-02-25.nc"
)
nc_cyc   = xr.open_dataset("cyclonic.nc")
nc_anti  = xr.open_dataset("anticyclonic.nc")
coast = gpd.read_file("ne_10m_coastline.shp")
coast = coast.to_crs("EPSG:4326")
nc_ekman=xr.open_dataset("ekman_daily_outputs.nc")

ERROR 1: PROJ: proj_create_from_database: Open of /home/users/ltharun/.conda/envs/chl_forecast_clone/share/proj failed


In [18]:
datasets = {
    "SST": nc_sst,
    "Currents": nc_curr,
    "Chlorophyll": chl_nc,
    "CCA Magnitude": nc_cca_mag,
    "Wind": wind,
    "Depth": nc_depth,
    "SSS": nc_sss,
    "Cyclonic": nc_cyc,
    "Anticyclonic": nc_anti
}


In [19]:
for name, ds in datasets.items():
    print("=" * 80)
    print(f"{name}")
    print("=" * 80)
    print(ds)
    print("\n")

SST
<xarray.Dataset> Size: 17GB
Dimensions:       (time: 9178, latitude: 380, longitude: 620)
Coordinates:
  * latitude      (latitude) float32 2kB 5.025 5.075 5.125 ... 23.88 23.93 23.98
  * longitude     (longitude) float32 2kB 65.02 65.07 65.12 ... 95.92 95.97
  * time          (time) datetime64[ns] 73kB 1998-01-10 ... 2023-02-25
Data variables:
    analysed_sst  (time, latitude, longitude) float64 17GB ...
Attributes: (12/49)
    Conventions:                CF-1.4, Unidata Observation Dataset v1.0
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    IN NO EVENT SHALL DMI OR ITS REPRESENTATIVES ...
    creator_email:              myocean_po@dmi.dk
    ...                         ...
    summary:                    DMI Sea and Ice Surface Temperature analysis ...
    time_coverage_end:          20241232T000000Z
    time_covera

In [20]:
for name, ds in datasets.items():
    rename_dict = {}

    if "valid_time" in ds.coords:
        rename_dict["valid_time"] = "time"
    if "latitude" in ds.coords:
        rename_dict["latitude"] = "lat"
    if "longitude" in ds.coords:
        rename_dict["longitude"] = "lon"

    if rename_dict:
        datasets[name] = ds.rename(rename_dict)

In [21]:
datasets

{'SST': <xarray.Dataset> Size: 17GB
 Dimensions:       (time: 9178, lat: 380, lon: 620)
 Coordinates:
   * lat           (lat) float32 2kB 5.025 5.075 5.125 ... 23.88 23.93 23.98
   * lon           (lon) float32 2kB 65.02 65.07 65.12 ... 95.88 95.92 95.97
   * time          (time) datetime64[ns] 73kB 1998-01-10 ... 2023-02-25
 Data variables:
     analysed_sst  (time, lat, lon) float64 17GB ...
 Attributes: (12/49)
     Conventions:                CF-1.4, Unidata Observation Dataset v1.0
     Metadata_Conventions:       Unidata Dataset Discovery v1.0
     acknowledgment:             Please acknowledge the use of these data with...
     cdm_data_type:              grid
     comment:                    IN NO EVENT SHALL DMI OR ITS REPRESENTATIVES ...
     creator_email:              myocean_po@dmi.dk
     ...                         ...
     summary:                    DMI Sea and Ice Surface Temperature analysis ...
     time_coverage_end:          20241232T000000Z
     time_coverage_st

In [22]:
nc_sst = datasets["SST"]
nc_curr = datasets["Currents"]
chl_nc = datasets["Chlorophyll"]
nc_cca_mag = datasets["CCA Magnitude"]
wind = datasets["Wind"]
nc_depth = datasets["Depth"]
nc_sss = datasets["SSS"]
nc_cyc = datasets["Cyclonic"]
nc_anti = datasets["Anticyclonic"]

In [23]:
## Trees
lat=nc_sst['lat'].values
lon=nc_sst['lon'].values
lon_grid,lat_grid=np.meshgrid(lon,lat)
points=np.column_stack((lat_grid.ravel(),lon_grid.ravel()))
tree=cKDTree(points)
shape=lat_grid.shape

In [24]:
def get_sst(date):
    try:
        da=nc_sst.sel(time=date,method='nearest')
        var=list(da.data_vars.keys())[0]
        sst=da[var].values
        if sst.ndim==3:
            sst=sst[0]
        if np.nanmax(sst)>200:
            sst=sst-273.15
        return sst.astype(float)
    except:
        return np.nan
def get_idx(lat,lon):
    _, idx=tree.query([lat,lon])
    return np.unravel_index(idx,shape)


In [25]:
def process_dataset(data):
    sst_list=[]
    grad_list=[]
    
    for i,row in data.iterrows():
        if i % 300==0:
            print(f'processing {i}/{len(data)}')
        date=row['Date']
        lat=row['Latitude']
        lon=row['Longitude']
        if np.isnan(lat) or np.isnan(lon):
            sst_list.append(np.nan)
            grad_list.append(np.nan)
            continue
        sst=get_sst(date)
        if sst is None:
            sst_list.append(np.nan)
            grad_list.append(np.nan)
            continue
        grad=gaussian_gradient_magnitude(np.nan_to_num(sst,nan=0),sigma=1)
        r,c=get_idx(lat,lon)
        sst_da = xr.DataArray(sst,coords=[nc_sst.lat.values, nc_sst.lon.values],dims=['lat','lon'])

        sst_val = float(sst_da.interp(lat=lat,lon=lon).values)

        sst_list.append(sst_val)
        grad_list.append(grad[r,c])
    data['SST']=sst_list
    data['SST_Grad']=grad_list
    return data

In [26]:
data=process_dataset(data)

processing 0/3577
processing 300/3577
processing 600/3577
processing 900/3577
processing 1200/3577
processing 1500/3577
processing 1800/3577
processing 2100/3577
processing 2400/3577
processing 2700/3577
processing 3000/3577
processing 3300/3577
processing 3600/3577


In [27]:
data2=process_dataset(data2)

processing 0/3421
processing 300/3421
processing 600/3421
processing 900/3421
processing 1200/3421
processing 1500/3421
processing 1800/3421
processing 2100/3421
processing 2400/3421
processing 2700/3421
processing 3000/3421
processing 3300/3421


In [28]:
def get_curr(date,lat,lon):
    try:
        try:
            da=nc_curr.sel(time=date)
        except:
            da=nc_curr.sel(time=date,method='nearest')
        uo=da['ugos']
        vo=da['vgos']
        if uo.ndim==3:
            uo=uo[0]
            vo=vo[0]
        uo_val=float(uo.interp(lat=lat,lon=lon).values)
        vo_val=float(vo.interp(lat=lat,lon=lon).values)
        return float(np.sqrt(uo_val**2+vo_val**2))
    except:
        return np.nan
def get_ssh(date,lat,lon):
    try:
        try:
            da=nc_curr.sel(time=date)
        except:
            da=nc_curr.sel(time=date,method='nearest')
        ssh=da['sla']
        if ssh.ndim==3:
            ssh=ssh[0]
        ssh_val=float(ssh.interp(lat=lat,lon=lon).values)
        return ssh_val
    except:
        return np.nan
        

In [29]:
def process_curr(data):
    curr_speed=[]
    ssh_list=[]
    for i,row in data.iterrows():
        if i % 300==0:
            print(f'processing {i}/{len(data)}')
        date=row['Date']
        lat=row['Latitude']
        lon=row['Longitude']
        if np.isnan(lat) or np.isnan(lon):
            curr_speed.append(np.nan)
            ssh_list.append(np.nan)
            continue
        speed=get_curr(date,lat,lon)
        ssh=get_ssh(date,lat,lon)
        curr_speed.append(speed)
        ssh_list.append(ssh)
    data['curr_Speed']=curr_speed
    data['SSH']=ssh_list
    return data

In [30]:
data=process_curr(data)


processing 0/3577
processing 300/3577
processing 600/3577
processing 900/3577
processing 1200/3577
processing 1500/3577
processing 1800/3577
processing 2100/3577
processing 2400/3577
processing 2700/3577
processing 3000/3577
processing 3300/3577
processing 3600/3577


In [31]:
data2=process_curr(data2)

processing 0/3421
processing 300/3421
processing 600/3421
processing 900/3421
processing 1200/3421
processing 1500/3421
processing 1800/3421
processing 2100/3421
processing 2400/3421
processing 2700/3421
processing 3000/3421
processing 3300/3421


In [32]:
data

,Date,Latitude,Longitude,Total,SST,SST_Grad,curr_Speed,SSH
0,2009-02-19,18.455278,84.431111,390.00,27.466531,0.363524,0.312740,-0.036798
1,2009-02-14,18.445000,84.494167,1000.00,26.903163,0.056516,0.377638,-0.083843
2,2009-02-19,18.240833,84.161667,500.00,27.431073,0.221516,0.204339,-0.028932
3,2009-02-12,17.993611,83.774444,550.00,26.796256,0.094927,0.268371,-0.089219
4,2009-03-08,17.708056,83.390000,95.00,27.773549,6.265504,0.166809,0.033530
...,...,...,...,...,...,...,...,...
3684,2014-01-27,13.398150,74.588683,186.10,28.424005,0.206649,0.174194,0.160499
3685,2014-01-27,13.391500,74.579500,282.80,28.437495,0.206649,0.174665,0.160001
3686,2014-02-07,12.858325,74.727392,265.95,28.499733,0.368505,0.209361,0.144084
3687,2014-02-20,13.398150,74.588683,243.35,28.692647,0.281591,0.065364,0.125120


In [33]:
if chl_nc.lat.values[0] > chl_nc.lat.values[-1]:
    chl_nc = chl_nc.sortby("lat")

In [34]:
def get_chl(date,lat,lon):
    try:
        try:
            da=chl_nc.sel(time=date)
        except:
            da=chl_nc.sel(time=date, method='nearest')
        chl=da['CHL']
        if chl.ndim==3:
            chl=chl[0]
        chl_val=float(chl.interp(lat=lat, lon=lon).values)
        return chl_val
    except:
        return np.nan

In [35]:
def process(data):
    CHL=[]
    for i,row in data.iterrows():
        if i%300==0:
            print(i)
        date=row['Date']
        lat=row['Latitude']
        lon=row['Longitude']
        if np.isnan(lat) or np.isnan(lon):
            CHL.append(np.nan)
            continue
        chl=get_chl(date,lat,lon)
        CHL.append(chl)
    data['CHL']=CHL
    return data
            

In [36]:
data = process(data)


0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [37]:
data2 = process(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [38]:
data

,Date,Latitude,Longitude,Total,SST,SST_Grad,curr_Speed,SSH,CHL
0,2009-02-19,18.455278,84.431111,390.00,27.466531,0.363524,0.312740,-0.036798,0.806467
1,2009-02-14,18.445000,84.494167,1000.00,26.903163,0.056516,0.377638,-0.083843,0.426134
2,2009-02-19,18.240833,84.161667,500.00,27.431073,0.221516,0.204339,-0.028932,1.443785
3,2009-02-12,17.993611,83.774444,550.00,26.796256,0.094927,0.268371,-0.089219,0.473846
4,2009-03-08,17.708056,83.390000,95.00,27.773549,6.265504,0.166809,0.033530,0.976604
...,...,...,...,...,...,...,...,...,...
3684,2014-01-27,13.398150,74.588683,186.10,28.424005,0.206649,0.174194,0.160499,1.155105
3685,2014-01-27,13.391500,74.579500,282.80,28.437495,0.206649,0.174665,0.160001,1.084983
3686,2014-02-07,12.858325,74.727392,265.95,28.499733,0.368505,0.209361,0.144084,2.479993
3687,2014-02-20,13.398150,74.588683,243.35,28.692647,0.281591,0.065364,0.125120,1.137792


In [39]:

def get_front_magnitude(date, lat, lon):

    try:
        try:
            da = nc_cca_mag.sel(time=date)
        except:
            da = nc_cca_mag.sel(time=date, method="nearest")

        mask = da["CCA"].values == 1

        if not np.any(mask):
            return np.nan

        rows, cols = np.where(mask)

        front_lat = da["lat"].values[rows]
        front_lon = da["lon"].values[cols]
        front_mag = da["delta_T"].values[rows, cols]

        # Remove invalid values
        valid = (
            ~np.isnan(front_lat) &
            ~np.isnan(front_lon) &
            ~np.isnan(front_mag)
        )

        front_lat = front_lat[valid]
        front_lon = front_lon[valid]
        front_mag = front_mag[valid]

        if len(front_lat) == 0:
            return np.nan

        coords = np.deg2rad(np.column_stack((front_lat, front_lon)))
        tree = BallTree(coords, metric="haversine")

        _, ind = tree.query(np.deg2rad([[lat, lon]]), k=1)

        nearest_magnitude = front_mag[ind[0][0]]

        return nearest_magnitude

    except Exception as e:
        print(e)
        return np.nan

In [40]:
def process_front_info(df):

    MAG = []

    for i, row in df.iterrows():

        if i % 300 == 0:
            print(i)

        mag = get_front_magnitude(
            row["Date"],
            row["Latitude"],
            row["Longitude"]
        )

        MAG.append(mag)

    df["Nearest_Front_Magnitude"] = MAG

    return df

In [68]:
data=process_front_info(data)


0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [81]:
data2=process_front_info(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [43]:
def get_wind(date, lat, lon):
    try:
        try:
            da = wind.sel(time=date)
        except:
            da = wind.sel(time=date, method="nearest")

        u = da["u10"]
        v = da["v10"]

        if u.ndim == 3:
            u = u[0]
            v = v[0]

        wind_speed = np.sqrt(u**2 + v**2)

        return float(
            wind_speed.interp(lat=lat, lon=lon).values
        )

    except:
        return np.nan

In [44]:
def process_wind(data):

    WIND = []

    for i, row in data.iterrows():

        if i % 300 == 0:
            print(i)

        date = row["Date"]
        lat = row["Latitude"]
        lon = row["Longitude"]

        if np.isnan(lat) or np.isnan(lon):
            WIND.append(np.nan)
            continue

        wind = get_wind(date, lat, lon)
        WIND.append(wind)

    data["WIND_SPEED"] = WIND

    return data

In [45]:
data=process_wind(data)


0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [46]:
data2=process_wind(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [47]:
def get_depth(lat, lon):

    try:
        depth = nc_depth["elevation"].interp(
            lat=lat,
            lon=lon
        )

        # Ocean depth should be positive
        return float(-depth.values)

    except:
        return np.nan

In [48]:
def process_depth(df):

    DEPTH = []

    for i, row in df.iterrows():

        if i % 300 == 0:
            print(i)

        lat = row["Latitude"]
        lon = row["Longitude"]

        if np.isnan(lat) or np.isnan(lon):
            DEPTH.append(np.nan)
            continue

        DEPTH.append(
            get_depth(
                lat,
                lon
            )
        )

    df["Depth"] = DEPTH

    return df

In [49]:
data=process_depth(data)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [50]:
data2=process_depth(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [69]:
nc_sss

<xarray.Dataset> Size: 3GB
Dimensions:  (time: 9187, depth: 1, lat: 166, lon: 233)
Coordinates:
  * depth    (depth) float32 4B 0.0
  * lat      (lat) float32 664B 2.438 2.562 2.688 2.812 ... 22.81 22.94 23.06
  * lon      (lon) float32 932B 66.56 66.69 66.81 66.94 ... 95.31 95.44 95.56
  * time     (time) datetime64[ns] 73kB 1998-01-01 1998-01-02 ... 2023-02-25
Data variables:
    sos      (time, depth, lat, lon) float64 3GB ...
Attributes: (12/17)
    Conventions:               CF-1.7
    Scaling_Equation:          (scale_factor*data) + add_offset
    contact:                   servicedesk.cmems@mercator-ocean.eu
    creation_date:             Sun 13 Aug 2023 18:24:41
    easternmost_longitude:     359.9375
    grid_resolution:           0.125 degrees
    ...                        ...
    references:                Buongiorno Nardelli, B., R. Droghei, and R. Sa...
    software_version:          SSS/SSD HR Processor v1.0
    southernmost_latitude:     -89.9375
    title:                     Global Analysed Sea Surface Salinity and Density
    westernmost_longitude:     0.0625
    copernicusmarine_version:  2.3.0

In [76]:
def get_sss(date, lat, lon):

    try:
        sss = nc_sss["sos"].sel(
            time=date,
            lat=lat,
            lon=lon,
            method="nearest"
        )

        return float(sss.values)

    except:
        return np.nan

In [77]:
def process_sss(df):

    SSS = []

    for i, row in df.iterrows():

        if i % 300 == 0:
            print(i)

        date = row["Date"]
        lat = row["Latitude"]
        lon = row["Longitude"]

        if (
            pd.isna(date)
            or np.isnan(lat)
            or np.isnan(lon)
        ):
            SSS.append(np.nan)
            continue

        SSS.append(
            get_sss(
                date,
                lat,
                lon
            )
        )

    df["SSS"] = SSS

    return df

In [78]:
data=process_sss(data)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [79]:
data2=process_sss(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [55]:
coastline = coast.unary_union

In [56]:
def get_distance_to_coast(lat, lon):

    try:

        point = Point(lon, lat)

        nearest = nearest_points(point, coastline)[1]

        return geodesic(
            (lat, lon),
            (nearest.y, nearest.x)
        ).km

    except:
        return np.nan

In [57]:
def process_distance_to_coast(df):

    DIST = []

    for i, row in df.iterrows():

        if i % 300 == 0:
            print(i)

        lat = row["Latitude"]
        lon = row["Longitude"]

        if np.isnan(lat) or np.isnan(lon):

            DIST.append(np.nan)
            continue

        DIST.append(
            get_distance_to_coast(
                lat,
                lon
            )
        )

    df["Distance_to_Coast_km"] = DIST

    return df

In [58]:
data=process_distance_to_coast(data)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [59]:
data2=process_distance_to_coast(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [89]:
nc_eddy = xr.concat(
    [nc_cyc, nc_anti],
    dim="obs"
)

# Add the eddy type
nc_eddy["Type"] = (
    "obs",
    np.concatenate([
        np.ones(nc_cyc.dims["obs"], dtype=np.int8),
        -np.ones(nc_anti.dims["obs"], dtype=np.int8)
    ])
)

In [90]:
from collections import defaultdict

date_index = defaultdict(list)

times = nc_eddy["time"].values.astype("datetime64[D]")

for i, t in enumerate(times):
    date_index[t].append(i)

In [91]:
from shapely.geometry import Polygon
import numpy as np

contour_lon = nc_eddy["effective_contour_longitude"].values
contour_lat = nc_eddy["effective_contour_latitude"].values

polygon_cache = {}

for i in range(len(contour_lon)):

    lons = contour_lon[i]
    lats = contour_lat[i]

    mask = (~np.isnan(lons)) & (~np.isnan(lats))

    lons = lons[mask]
    lats = lats[mask]

    if len(lons) >= 3:
        polygon_cache[i] = Polygon(zip(lons, lats))

In [92]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    dlat = radians(lat2-lat1)
    dlon = radians(lon2-lon1)

    a = (sin(dlat/2)**2 +
         cos(radians(lat1)) *
         cos(radians(lat2)) *
         sin(dlon/2)**2)

    return 2 * R * atan2(sqrt(a), sqrt(1-a))

In [93]:
from shapely.geometry import Point
import numpy as np
import pandas as pd

def get_eddy_features(date, lat, lon):

    date = np.datetime64(pd.to_datetime(date).normalize())

    point = Point(lon, lat)

    inside = 0
    nearest_dist = np.inf

    for obs in date_index.get(date, []):

        # Check if point is inside the eddy
        if polygon_cache[obs].contains(point):
            inside = 1

        # Distance to eddy centre
        d = haversine(
            lat,
            lon,
            nc_eddy["latitude"].values[obs],
            nc_eddy["longitude"].values[obs]
        )

        if d < nearest_dist:
            nearest_dist = d

    return inside, nearest_dist

In [94]:
def process_eddy(df):

    inside = []
    dist = []

    for i, row in df.iterrows():

        if i % 300 == 0:
            print(i)

        x = get_eddy_features(
            row["Date"],
            row["Latitude"],
            row["Longitude"]
        )

        inside.append(x[0])
        dist.append(x[1])

    df["Inside_Eddy"] = inside
    df["Nearest_Eddy_Distance_km"] = dist

    return df

In [95]:
data=process_eddy(data)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [97]:
data2=process_eddy(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [172]:
nc_ekman=xr.open_dataset('ekman_daily_outputs.nc')

In [174]:
def get_ekman(date, lat, lon):

    try:
        try:
            da = nc_ekman.sel(time=date)
        except:
            da = nc_ekman.sel(time=date, method="nearest")

        mx = da["M_x"]
        my = da["M_y"]
        wek = da["wek_m_s"]

        # Remove singleton time dimension if present
        if mx.ndim == 3:
            mx = mx[0]
            my = my[0]
            wek = wek[0]

        mx_val = float(mx.interp(lat=lat, lon=lon).values)
        my_val = float(my.interp(lat=lat, lon=lon).values)
        wek_val = float(wek.interp(lat=lat, lon=lon).values)

        return mx_val, my_val, wek_val

    except Exception as e:
        print(e)
        return np.nan, np.nan, np.nan


def process_ekman(data):

    MX = []
    MY = []
    WEK = []

    for i, row in data.iterrows():

        if i % 300 == 0:
            print(i)

        date = row["Date"]
        lat = row["Latitude"]
        lon = row["Longitude"]

        if np.isnan(lat) or np.isnan(lon):
            MX.append(np.nan)
            MY.append(np.nan)
            WEK.append(np.nan)
            continue

        mx, my, wek = get_ekman(date, lat, lon)

        MX.append(mx)
        MY.append(my)
        WEK.append(wek)

    data["Ekman_X"] = MX
    data["Ekman_Y"] = MY
    data["Ekman_Pumping"] = WEK

    return data

In [178]:
if 'latitude' in nc_ekman.coords:
    nc_ekman=nc_ekman.rename({'latitude':'lat','longitude':'lon','valid_time':'time'}) 

In [179]:
data= process_ekman(data)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600


In [180]:
data2= process_ekman(data2)

0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300


In [177]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3577 entries, 0 to 3688
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      3577 non-null   datetime64[ns]
 1   Latitude                  3577 non-null   float64       
 2   Longitude                 3577 non-null   float64       
 3   Total                     3557 non-null   float64       
 4   SST                       3553 non-null   float64       
 5   SST_Grad                  3577 non-null   float64       
 6   curr_Speed                3526 non-null   float64       
 7   SSH                       3541 non-null   float64       
 8   CHL                       3477 non-null   float64       
 9   Nearest_Front_Magnitude   3577 non-null   float64       
 10  WIND_SPEED                3577 non-null   float64       
 11  Depth                     3577 non-null   float64       
 12  SSS                      

In [98]:
missing_percent = (data.isna().mean() * 100).sort_values(ascending=False)

print(missing_percent)

CHL                         2.795639
SSS                         1.873078
curr_Speed                  1.425776
SSH                         1.006430
SST                         0.670953
Total                       0.559128
Longitude                   0.000000
Latitude                    0.000000
SST_Grad                    0.000000
Date                        0.000000
WIND_SPEED                  0.000000
Nearest_Front_Magnitude     0.000000
Depth                       0.000000
Distance_to_Coast_km        0.000000
Inside_Eddy                 0.000000
Nearest_Eddy_Distance_km    0.000000
dtype: float64


In [191]:
merged = pd.concat([data, data2], ignore_index=True)

In [192]:
merged["Date"] = pd.to_datetime(merged["Date"], errors="coerce")

In [193]:
duplicates = merged.duplicated(
    subset=["Date", "Latitude", "Longitude"]
)

print("Number of duplicates:", duplicates.sum())

Number of duplicates: 1823


In [194]:
group=['Date', 'Latitude', 'Longitude']

In [195]:
data.columns

Index(['Date', 'Latitude', 'Longitude', 'Total', 'SST', 'SST_Grad',
       'curr_Speed', 'SSH', 'CHL', 'Nearest_Front_Magnitude', 'WIND_SPEED',
       'Depth', 'SSS', 'Distance_to_Coast_km', 'Inside_Eddy',
       'Nearest_Eddy_Distance_km', 'Ekman_X', 'Ekman_Y', 'Ekman_Pumping'],
      dtype='object')

In [196]:
agg = {
    "Total": "sum",

    "SST": "mean",
    "SST_Grad": "mean",
    "WIND_SPEED": "mean",
    "curr_Speed": "mean",
    "SSH": "mean",
    "CHL": "mean",
    "WIND_SPEED": "mean",
    "Nearest_Front_Magnitude": "mean",
    "Depth": "mean",
    "SSS": "mean",

    "Distance_to_Coast_km": "mean",

    "Inside_Eddy": "mean",
    "Nearest_Eddy_Distance_km": "mean",
    "Ekman_Pumping": "mean",
    "Ekman_X": "mean",
    "Ekman_Y": "mean"
}

In [197]:
merged=merged.groupby(group,as_index=False).agg(agg)

In [198]:
merged

,Date,Latitude,Longitude,Total,SST,SST_Grad,WIND_SPEED,curr_Speed,SSH,CHL,Nearest_Front_Magnitude,Depth,SSS,Distance_to_Coast_km,Inside_Eddy,Nearest_Eddy_Distance_km,Ekman_Pumping,Ekman_X,Ekman_Y
0,1998-01-13,17.750000,84.200000,63.0,26.625002,0.138237,4.391017,0.621994,-0.026420,0.197359,0.854675,1813.50,31.576618,57.937339,0.0,143.850021,1.019260e-06,-0.524193,0.600235
1,1998-01-14,17.716667,85.366667,234.0,26.427218,0.079969,4.313646,0.206300,0.130808,0.209597,0.852051,2521.75,31.647762,141.926745,1.0,130.819779,-6.194176e-07,-0.355248,0.683647
2,1998-01-15,17.583333,86.166667,167.0,26.596661,0.081181,3.388563,0.293822,0.115314,0.173928,0.640825,2547.00,31.785763,216.721904,1.0,77.782322,1.183340e-06,-0.068971,0.473820
3,1998-01-16,17.666667,86.766667,57.0,26.563610,0.157519,4.244010,0.225370,0.113047,0.166628,0.694055,2494.75,31.938530,251.398372,1.0,59.102452,3.152014e-06,-0.495753,0.561133
4,1998-01-17,17.683333,87.450000,43.0,26.818327,0.097953,3.575337,0.196991,0.077844,0.158637,0.559991,2365.25,31.798199,276.213027,1.0,100.732519,7.607056e-07,-0.437262,0.301395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5170,2023-02-16,12.500000,94.166667,207.0,28.253325,0.010053,5.214817,0.140866,0.129308,0.113029,0.522272,2238.00,32.400047,40.925896,1.0,96.327419,3.244443e-06,0.039161,-1.582394
5171,2023-02-21,11.466667,93.166667,220.0,28.348872,0.072470,5.160966,0.127026,0.115900,0.134514,0.443405,1701.25,32.725090,46.887755,0.0,83.359735,3.830412e-06,1.360474,-0.999035
5172,2023-02-22,12.133333,93.316667,80.0,28.244162,0.017804,4.480051,0.277792,0.079379,0.154524,0.629684,958.50,32.716763,23.383443,0.0,52.058389,1.106589e-06,0.881620,-0.820103
5173,2023-02-23,12.133333,93.650000,251.0,28.167493,0.010607,5.237947,0.331476,0.067562,0.156496,0.468257,1444.50,32.618679,26.264593,0.0,38.783148,2.863342e-06,1.443780,-0.787623


In [199]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5175 entries, 0 to 5174
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      5175 non-null   datetime64[ns]
 1   Latitude                  5175 non-null   float64       
 2   Longitude                 5175 non-null   float64       
 3   Total                     5175 non-null   float64       
 4   SST                       5146 non-null   float64       
 5   SST_Grad                  5175 non-null   float64       
 6   WIND_SPEED                5175 non-null   float64       
 7   curr_Speed                5119 non-null   float64       
 8   SSH                       5134 non-null   float64       
 9   CHL                       5068 non-null   float64       
 10  Nearest_Front_Magnitude   5175 non-null   float64       
 11  Depth                     5175 non-null   float64       
 12  SSS                 

In [200]:
df=merged.copy()

In [201]:
df=df.dropna(subset=['Date', 'Latitude', 'Longitude', 'Total'])

In [202]:
df

,Date,Latitude,Longitude,Total,SST,SST_Grad,WIND_SPEED,curr_Speed,SSH,CHL,Nearest_Front_Magnitude,Depth,SSS,Distance_to_Coast_km,Inside_Eddy,Nearest_Eddy_Distance_km,Ekman_Pumping,Ekman_X,Ekman_Y
0,1998-01-13,17.750000,84.200000,63.0,26.625002,0.138237,4.391017,0.621994,-0.026420,0.197359,0.854675,1813.50,31.576618,57.937339,0.0,143.850021,1.019260e-06,-0.524193,0.600235
1,1998-01-14,17.716667,85.366667,234.0,26.427218,0.079969,4.313646,0.206300,0.130808,0.209597,0.852051,2521.75,31.647762,141.926745,1.0,130.819779,-6.194176e-07,-0.355248,0.683647
2,1998-01-15,17.583333,86.166667,167.0,26.596661,0.081181,3.388563,0.293822,0.115314,0.173928,0.640825,2547.00,31.785763,216.721904,1.0,77.782322,1.183340e-06,-0.068971,0.473820
3,1998-01-16,17.666667,86.766667,57.0,26.563610,0.157519,4.244010,0.225370,0.113047,0.166628,0.694055,2494.75,31.938530,251.398372,1.0,59.102452,3.152014e-06,-0.495753,0.561133
4,1998-01-17,17.683333,87.450000,43.0,26.818327,0.097953,3.575337,0.196991,0.077844,0.158637,0.559991,2365.25,31.798199,276.213027,1.0,100.732519,7.607056e-07,-0.437262,0.301395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5170,2023-02-16,12.500000,94.166667,207.0,28.253325,0.010053,5.214817,0.140866,0.129308,0.113029,0.522272,2238.00,32.400047,40.925896,1.0,96.327419,3.244443e-06,0.039161,-1.582394
5171,2023-02-21,11.466667,93.166667,220.0,28.348872,0.072470,5.160966,0.127026,0.115900,0.134514,0.443405,1701.25,32.725090,46.887755,0.0,83.359735,3.830412e-06,1.360474,-0.999035
5172,2023-02-22,12.133333,93.316667,80.0,28.244162,0.017804,4.480051,0.277792,0.079379,0.154524,0.629684,958.50,32.716763,23.383443,0.0,52.058389,1.106589e-06,0.881620,-0.820103
5173,2023-02-23,12.133333,93.650000,251.0,28.167493,0.010607,5.237947,0.331476,0.067562,0.156496,0.468257,1444.50,32.618679,26.264593,0.0,38.783148,2.863342e-06,1.443780,-0.787623


In [203]:
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

print(missing_percent)

CHL                         2.067633
SSS                         1.294686
curr_Speed                  1.082126
SSH                         0.792271
SST                         0.560386
Ekman_Pumping               0.038647
Ekman_Y                     0.019324
Ekman_X                     0.019324
Longitude                   0.000000
Date                        0.000000
Latitude                    0.000000
Nearest_Front_Magnitude     0.000000
Total                       0.000000
SST_Grad                    0.000000
WIND_SPEED                  0.000000
Inside_Eddy                 0.000000
Distance_to_Coast_km        0.000000
Depth                       0.000000
Nearest_Eddy_Distance_km    0.000000
dtype: float64


In [204]:
df['log_chl']=np.log1p(df['CHL'])

In [205]:
df['SST_X_CHL']=df['SST'] * df['log_chl']

In [206]:
df['wind_X_curr']=df['WIND_SPEED']*df['curr_Speed']

In [207]:
import numpy as np
df["Month"]=df['Date'].dt.month
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

In [208]:
import numpy as np
df["DayOfYear"] = df["Date"].dt.dayofyear
df["DOY_sin"] = np.sin(2 * np.pi * df["DayOfYear"] / 365.25)
df["DOY_cos"] = np.cos(2 * np.pi * df["DayOfYear"] / 365.25)

In [209]:
df = df.sort_values("Date")

In [210]:
features=[ 'SST', 'SST_Grad',
       'curr_Speed', 'SSH', 'CHL','WIND_SPEED',
       'Nearest_Front_Magnitude', 'Depth', 
       'log_chl', 'SST_X_CHL', 'wind_X_curr', 'DOY_sin', 'DOY_cos']

In [211]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

cols = ["SSS", "SST", "SSH", "CHL", "Depth"]

df[cols] = imputer.fit_transform(df[cols])

In [212]:
df = df.dropna(subset=features)

In [213]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5041 entries, 0 to 5174
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      5041 non-null   datetime64[ns]
 1   Latitude                  5041 non-null   float64       
 2   Longitude                 5041 non-null   float64       
 3   Total                     5041 non-null   float64       
 4   SST                       5041 non-null   float64       
 5   SST_Grad                  5041 non-null   float64       
 6   WIND_SPEED                5041 non-null   float64       
 7   curr_Speed                5041 non-null   float64       
 8   SSH                       5041 non-null   float64       
 9   CHL                       5041 non-null   float64       
 10  Nearest_Front_Magnitude   5041 non-null   float64       
 11  Depth                     5041 non-null   float64       
 12  SSS                      

In [214]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

# KMeans on log-transformed Total
X = np.log1p(df[["Total"]])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
df["Cluster"] = kmeans.fit_predict(X)

# View cluster centers
centers = np.expm1(kmeans.cluster_centers_.flatten())

cluster_info = pd.DataFrame({
    "Cluster": range(3),
    "Center": centers
}).sort_values("Center")

print(cluster_info)

   Cluster       Center
2        2     7.238786
0        0    96.597073
1        1  1283.096895


In [215]:
print(
    df.groupby("Cluster")["Total"].agg(
        ["min", "max", "mean", "median", "count"]
    )
)

           min      max         mean  median  count
Cluster                                            
0         28.0    353.0   121.745156   100.0   2911
1        354.0  20000.0  1932.497366  1190.0    949
2          0.0     27.0    10.936393    10.0   1181


In [216]:
mapping = {
    cluster_info.iloc[0]["Cluster"]: "Low",
    cluster_info.iloc[1]["Cluster"]: "Medium",
    cluster_info.iloc[2]["Cluster"]: "High"
}

df["Class"] = df["Cluster"].map(mapping)

print(
    df.groupby("Class")["Total"].agg(
        ["min", "max", "mean", "median", "count"]
    )
)

          min      max         mean  median  count
Class                                             
High    354.0  20000.0  1932.497366  1190.0    949
Low       0.0     27.0    10.936393    10.0   1181
Medium   28.0    353.0   121.745156   100.0   2911


In [245]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df[[
    'SST',
    'curr_Speed',
    'SSH',
    'Nearest_Front_Magnitude',
    'Depth',
    'SSS',
    'log_chl',
    'SST_X_CHL',
    'Month_sin',
    'Month_cos',
    'DOY_sin',
    'DOY_cos',
    'Distance_to_Coast_km',"WIND_SPEED",
    'Nearest_Eddy_Distance_km','Inside_Eddy', 'Ekman_X', 'Ekman_Y'
]]

le = LabelEncoder()
y = le.fit_transform(df["Class"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [257]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# -------------------------------
# EXTRA TREES CLASSIFIER
# -------------------------------
et = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

# Train
et.fit(X_train, y_train)

# Predict
pred = et.predict(X_test)

# -------------------------------
# EVALUATION
# -------------------------------
accuracy = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="macro")

print("\n=== EXTRA TREES CLASSIFIER ===")
print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {f1:.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    pred,
    target_names=le.classes_
))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))
from sklearn.metrics import balanced_accuracy_score

print(
    balanced_accuracy_score(y_test, pred)
)


=== EXTRA TREES CLASSIFIER ===
Accuracy : 0.7433
Macro F1 : 0.6977

Classification Report
              precision    recall  f1-score   support

        High       0.87      0.64      0.74       190
         Low       0.67      0.47      0.55       236
      Medium       0.74      0.89      0.80       583

    accuracy                           0.74      1009
   macro avg       0.76      0.67      0.70      1009
weighted avg       0.75      0.74      0.73      1009


Confusion Matrix
[[122   3  65]
 [  5 110 121]
 [ 13  52 518]]
0.6655715589231823


In [258]:
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# -------------------------------
# MODELS
# -------------------------------
models = {
    "EXTRA TREES CLASSIFIER": ExtraTreesClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "RANDOM FOREST CLASSIFIER": RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "XGBOOST CLASSIFIER": XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss"
    )
}

# -------------------------------
# TRAIN & COMPARE
# -------------------------------
results = []

for name, model in models.items():

    print("=" * 60)
    print(f"=== {name} ===")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}\n")

    print("Classification Report")
    print(classification_report(
        y_test,
        y_pred,
        target_names=le.classes_
    ))

    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    print(macro_f1)
    print()

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Macro F1": macro_f1
    })

# -------------------------------
# COMPARISON TABLE
# -------------------------------
import pandas as pd

results_df = pd.DataFrame(results)

print("=" * 60)
print("MODEL COMPARISON")
print(results_df.sort_values("Accuracy", ascending=False))

=== EXTRA TREES CLASSIFIER ===
Accuracy : 0.7433
Macro F1 : 0.6977

Classification Report
              precision    recall  f1-score   support

        High       0.87      0.64      0.74       190
         Low       0.67      0.47      0.55       236
      Medium       0.74      0.89      0.80       583

    accuracy                           0.74      1009
   macro avg       0.76      0.67      0.70      1009
weighted avg       0.75      0.74      0.73      1009


Confusion Matrix
[[122   3  65]
 [  5 110 121]
 [ 13  52 518]]
0.6976650577648084

=== RANDOM FOREST CLASSIFIER ===
Accuracy : 0.7235
Macro F1 : 0.6605

Classification Report
              precision    recall  f1-score   support

        High       0.84      0.60      0.70       190
         Low       0.64      0.39      0.48       236
      Medium       0.72      0.90      0.80       583

    accuracy                           0.72      1009
   macro avg       0.73      0.63      0.66      1009
weighted avg       0.72    

In [259]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    et,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring="f1_macro"
)

importance = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

print(importance)

Depth                       0.070065
SSS                         0.057894
Nearest_Eddy_Distance_km    0.054189
SST                         0.053613
DOY_cos                     0.039889
Month_cos                   0.034225
Distance_to_Coast_km        0.030958
Nearest_Front_Magnitude     0.030553
SSH                         0.024757
SST_X_CHL                   0.021118
log_chl                     0.019846
DOY_sin                     0.019606
WIND_SPEED                  0.019583
curr_Speed                  0.017470
Month_sin                   0.016905
Ekman_Y                     0.016148
Inside_Eddy                 0.014318
Ekman_X                     0.013101
dtype: float64


In [260]:
import os
import joblib

# Create models directory
os.makedirs("models", exist_ok=True)

# Save trained model
joblib.dump(et, "models/extra_trees_pfz_model.pkl")

# Save Label Encoder
joblib.dump(le, "models/label_encoder.pkl")

# Save feature list
FEATURES = [
    'SST',
    'curr_Speed',
    'SSH',
    'Nearest_Front_Magnitude',
    'Depth',
    'SSS',
    'log_chl',
    'SST_X_CHL',
    'Month_sin',
    'Month_cos',
    'DOY_sin',
    'DOY_cos',
    'Distance_to_Coast_km',"WIND_SPEED",
    'Nearest_Eddy_Distance_km','Inside_Eddy', 'Ekman_X', 'Ekman_Y'
    
]

joblib.dump(FEATURES, "models/features.pkl")

print("✅ Model saved successfully!")
print("📁 Saved files:")
print(" - models/extra_trees_pfz_model.pkl")
print(" - models/label_encoder.pkl")
print(" - models/features.pkl")

✅ Model saved successfully!
📁 Saved files:
 - models/extra_trees_pfz_model.pkl
 - models/label_encoder.pkl
 - models/features.pkl
